# Comparing BFS and A* Search

# Code

In [1]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import maze_helper as mh

#f = open("small_maze.txt", "r")
#f = open("medium_maze.txt", "r")
#f = open("large_maze.txt", "r")    # this has only one solution!
#f = open("open_maze.txt", "r")
#f = open("empty_maze.txt", "r")
#f = open("empty_maze_2.txt", "r")
#f = open("loops_maze.txt", "r")
f = open("L_maze.txt", "r")

maze_str = f.read()
maze = mh.parse_maze(maze_str)

In [2]:
# tree_search_solution.py has my actual implementation (not published)
import tree_search_solution as ts


# order in which we add new states to the frontier
ts.set_order("NESW")
#ts.set_order(random=True)

## BFS

Breadth-first search is an _optimal_ algorithm. BFS is an _uninformed search algorithm and has no idea where the goal is. It expands the search in concentric circles around the start till it hits the goal.

In [3]:
%time result = ts.best_first_search(maze, strategy = "BFS", debug = False, vis = False, animation = True)

ts.animate_maze(result)

CPU times: user 6.73 ms, sys: 0 ns, total: 6.73 ms
Wall time: 6.71 ms
Path length: 16
Reached squares: 155
Action sequence: ['E', 'E', 'E', 'E', 'S', 'E', 'E', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'E']


Note that BFS explores almost the whole maze. BFS keeps the whole tree (notes are represented by gray squares) in memory. This is a problem for many AI problems with large state spaces. 

## A* Search - Euclidean Heuristic

A* is an informed search algorithm using f(n) = g(n) + h(n). With an admissible heuristic, best-cost updates and reopening when needed, it finds a minimum-cost path. Euclidean and Manhattan are both admissible and consistent for this four-direction, unit-cost maze.


In [4]:
# set heuristic for my A* search implementation
#ts.heuristic = ts.manhattan
ts.heuristic = ts.euclidean

%time result = ts.best_first_search(maze, strategy = "A*", debug = False, vis = False, animation = True)
ts.animate_maze(result)

CPU times: user 1.94 ms, sys: 0 ns, total: 1.94 ms
Wall time: 1.94 ms
Path length: 16
Reached squares: 89
Action sequence: ['E', 'E', 'E', 'E', 'S', 'E', 'E', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'E', 'N']


## A* Search - Manhattan Heuristic

In [5]:
# set heuristic for my A* search implementation
ts.heuristic = ts.manhattan
#ts.heuristic = ts.euclidean

%time result = ts.best_first_search(maze, strategy = "A*", debug = False, vis = False, animation = True)
ts.animate_maze(result)

CPU times: user 1.27 ms, sys: 8 us, total: 1.28 ms
Wall time: 1.29 ms
Path length: 16
Reached squares: 69
Action sequence: ['E', 'E', 'E', 'E', 'S', 'E', 'E', 'E', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N']


## Comparison

* BFS and A* find shortest paths in this unit-cost setting.
* The original notebook reports 142 explored states. Counts depend on the implementation, tie-breaking and the definition of explored; the reproducible measurement for the current version is recorded below.
* Manhattan is generally more informative than Euclidean on a four-direction grid.
* A* still stores frontier/reached information and can need substantial memory. Weighted A* can trade path quality for search effort.


# Bài 5 — Bảng so sánh trực tiếp BFS và A*

In [6]:
from time import perf_counter
from statistics import median
import pandas as pd
from IPython.display import display

ts.set_order("NESW", random=False)
comparison_b5 = []
for label, strategy, h in [
    ("BFS", "BFS", ts.manhattan),
    ("A* — Euclidean", "A*", ts.euclidean),
    ("A* — Manhattan", "A*", ts.manhattan),
]:
    ts.heuristic = h
    result_b5 = ts.best_first_search(maze, strategy=strategy)
    durations_b5 = []
    for _ in range(11):
        begin_b5 = perf_counter()
        ts.best_first_search(maze, strategy=strategy)
        durations_b5.append((perf_counter() - begin_b5) * 1000)
    comparison_b5.append({
        "Algorithm": label, "Path cost": result_b5["path_cost"],
        "Expanded": result_b5["expanded"], "Reached": len(result_b5["reached"]),
        "Max depth": result_b5["max_depth"], "Max frontier": result_b5["max_frontier"],
        "Median ms": median(durations_b5),
    })
df_comparison_b5 = pd.DataFrame(comparison_b5)
assert df_comparison_b5["Path cost"].nunique() == 1
assert df_comparison_b5["Path cost"].iloc[0] == 16
display(df_comparison_b5.round(3))

,Algorithm,Path cost,Expanded,Reached,Max depth,Max frontier,Median ms
0,BFS,16,151,155,17,17,0.630
1,A* — Euclidean,16,62,89,16,27,0.424
2,A* — Manhattan,16,40,69,16,29,0.327


### Nhận xét

Cả ba tìm đường **16 bước** trên L_maze. BFS mở rộng **151** node, A* Euclidean **62**, A* Manhattan **40**. Hai heuristic giúp giảm vùng tìm kiếm; Manhattan giảm khoảng **73.5%** expansion so với BFS. Các giá trị này là kết quả của cài đặt hiện tại, khác số 142 trong ghi chú ví dụ gốc.

Trên lưới 4 hướng cost 1, Manhattan ≥ Euclidean và cả hai không vượt chi phí còn lại thật. Với h consistent và cập nhật g đúng, A* tìm được lời giải tối ưu.